<a href="https://colab.research.google.com/github/carlosforeroudea/ps1-g2/blob/main/01_eda_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Análisis exploratorio y preparación del dataset

### Objetivo

Realizar un análisis exploratorio del conjunto de datos `ccdv/arxiv-summarization`, con el propósito de caracterizar su estructura, calidad y extensión textual. Este análisis permitirá identificar posibles registros inválidos o duplicados, comprender la distribución de los artículos y sus resúmenes, y establecer las condiciones para la selección de una muestra experimental reproducible de al menos 300 artículos.

## 1. Carga y estructura del dataset

En esta sección se carag el dataset y se revisan las particiones disponibles en el conjunto de datos, el número de registros de cada una y las variables que componen los registros.

In [ ]:
!pip install -q datasets

In [ ]:
import pandas as pd

In [ ]:
#cargamos el dataset
from datasets import load_dataset

dataset = load_dataset("ccdv/arxiv-summarization")

El conjunto de datos contiene 215.913 registros distribuidos en tres particiones. La mayor parte corresponde al conjunto de entrenamiento (`train`), que representa aproximadamente el 94 % del total, mientras que `validation` y `test` contienen aproximadamente el 3 % cada uno.

In [ ]:
#vemos la cantidad de registros
resumen_particiones = pd.DataFrame({
    "Partición": ["train", "validation", "test"],
    "Cantidad de registros": [
        len(dataset["train"]),
        len(dataset["validation"]),
        len(dataset["test"])
    ]
})

resumen_particiones.loc[len(resumen_particiones)] = [
    "Total",
    resumen_particiones["Cantidad de registros"].sum()
]

resumen_particiones

,Partición,Cantidad de registros
0,train,203037
1,validation,6436
2,test,6440
3,Total,215913


### Variables del dataset

La inspección de las características del conjunto de datos muestra que cada registro contiene dos variables principales:

- **`article`**: texto correspondiente al contenido del artículo científico.
- **`abstract`**: resumen asociado al artículo.

La versión utilizada del dataset no contiene una variable identificadora `id`. Por esta razón, la correspondencia entre el artículo y su resumen se analiza mediante la asociación directa de ambos campos dentro de cada registro y su posición en la partición correspondiente.

In [ ]:
for particion in dataset:
    print(f"\n--- {particion.upper()} ---")
    print(dataset[particion].features)


--- TRAIN ---
{'article': Value('string'), 'abstract': Value('string')}

--- VALIDATION ---
{'article': Value('string'), 'abstract': Value('string')}

--- TEST ---
{'article': Value('string'), 'abstract': Value('string')}


In [ ]:
print("Columnas disponibles:")
print(dataset["train"].column_names)

Columnas disponibles:
['article', 'abstract']


## 2. Análisis de calidad de los datos

Se verificaron diferentes condiciones que podrían afectar el uso del conjunto de datos en las etapas posteriores: valores nulos, textos vacíos, registros duplicados y tipos de datos.

### 2.1 Valores nulos

Se verificó la presencia de valores nulos en los campos `article` y `abstract` para las tres particiones.

In [ ]:
for particion in dataset:
    df = dataset[particion].to_pandas()

    print(f"\n--- {particion.upper()} ---")
    print(df.isnull().sum())


--- TRAIN ---
article     0
abstract    0
dtype: int64

--- VALIDATION ---
article     0
abstract    0
dtype: int64

--- TEST ---
article     0
abstract    0
dtype: int64


No se encontraron valores nulos en ninguna de las variables analizadas en las particiones `train`, `validation` y `test`. Esto indica que no existen registros con valores ausentes representados explícitamente como nulos en los campos principales.

### 2.2 Registros vacíos

Se verificó si alguno de los campos contenía cadenas vacías o únicamente espacios en blanco, debido a que estos casos pueden representar registros sin contenido útil para el proceso de resumen.

In [ ]:
for particion in dataset:
    df = dataset[particion].to_pandas()

    articulos_vacios = df["article"].fillna("").str.strip().eq("").sum()
    abstracts_vacios = df["abstract"].fillna("").str.strip().eq("").sum()

    print(f"\n--- {particion.upper()} ---")
    print(f"Artículos vacíos: {articulos_vacios}")
    print(f"Abstracts vacíos: {abstracts_vacios}")


--- TRAIN ---
Artículos vacíos: 120
Abstracts vacíos: 3

--- VALIDATION ---
Artículos vacíos: 0
Abstracts vacíos: 0

--- TEST ---
Artículos vacíos: 0
Abstracts vacíos: 0


Se identificaron 120 artículos vacíos y 3 abstracts vacíos en la partición `train`. No se encontraron registros vacíos en `validation` ni `test`.

### 2.3 Registros duplicados

Se analizaron los registros duplicados a nivel de registro completo y, adicionalmente, se revisó la repetición independiente de los campos `article` y `abstract`.

In [ ]:
for particion in dataset:
    df = dataset[particion].to_pandas()

    duplicados = df.duplicated().sum()

    print(f"{particion}: {duplicados} registros duplicados")

train: 8 registros duplicados
validation: 0 registros duplicados
test: 0 registros duplicados


In [ ]:
for particion in dataset:
    df = dataset[particion].to_pandas()

    duplicados_article = df["article"].duplicated().sum()
    duplicados_abstract = df["abstract"].duplicated().sum()

    print(f"\n--- {particion.upper()} ---")
    print(f"Artículos duplicados: {duplicados_article}")
    print(f"Abstracts duplicados: {duplicados_abstract}")


--- TRAIN ---
Artículos duplicados: 135
Abstracts duplicados: 60

--- VALIDATION ---
Artículos duplicados: 0
Abstracts duplicados: 0

--- TEST ---
Artículos duplicados: 0
Abstracts duplicados: 0


En `train` se identificaron 8 registros completamente duplicados, mientras que no se encontraron duplicados completos en `validation` ni `test`.

También se identificaron 135 artículos y 60 abstracts repetidos en `train`.

### 2.4 Tipos de datos

Las variables `article` y `abstract` presentan el tipo de dato texto (`string` en el dataset de Hugging Face y `object` al convertir temporalmente los registros a pandas).

In [ ]:
for particion in dataset:
    df = dataset[particion].to_pandas()

    print(f"\n--- {particion.upper()} ---")
    print(df.dtypes)


--- TRAIN ---
article     object
abstract    object
dtype: object

--- VALIDATION ---
article     object
abstract    object
dtype: object

--- TEST ---
article     object
abstract    object
dtype: object


## 3. Análisis de extensión textual

Debido a que el proyecto estudia diferentes estrategias para el manejo de documentos que pueden superar las ventanas de contexto de los modelos preentrenados, se analizó la extensión de los artículos y sus respectivos abstracts.

Para cada partición se calculó la longitud promedio en caracteres y palabras.

In [ ]:
for particion in dataset:
    ds = dataset[particion]

    total_article_chars = 0
    total_abstract_chars = 0
    total_article_words = 0
    total_abstract_words = 0

    n = len(ds)

    for fila in ds:
        article = fila["article"]
        abstract = fila["abstract"]

        total_article_chars += len(article)
        total_abstract_chars += len(abstract)

        total_article_words += len(article.split())
        total_abstract_words += len(abstract.split())

    print(f"\n--- {particion.upper()} ---")

    print("Longitud promedio del artículo:")
    print(f"  Caracteres: {total_article_chars / n:.2f}")
    print(f"  Palabras: {total_article_words / n:.2f}")

    print("Longitud promedio del abstract:")
    print(f"  Caracteres: {total_abstract_chars / n:.2f}")
    print(f"  Palabras: {total_abstract_words / n:.2f}")


--- TRAIN ---
Longitud promedio del artículo:
  Caracteres: 33841.54
  Palabras: 6038.19
Longitud promedio del abstract:
  Caracteres: 1619.36
  Palabras: 279.66

--- VALIDATION ---
Longitud promedio del artículo:
  Caracteres: 33028.41
  Palabras: 5894.43
Longitud promedio del abstract:
  Caracteres: 958.80
  Palabras: 161.54

--- TEST ---
Longitud promedio del artículo:
  Caracteres: 33062.25
  Palabras: 5905.87
Longitud promedio del abstract:
  Caracteres: 966.45
  Palabras: 163.13


### Resultados

Los artículos presentan una extensión considerablemente superior a la de sus respectivos abstracts. En `train`, por ejemplo, un artículo contiene en promedio 6.038 palabras, mientras que su abstract contiene aproximadamente 280 palabras.

Esta diferencia evidencia una relación aproximada de 21,6 a 1 entre la extensión promedio del artículo y la del resumen en el conjunto de entrenamiento.

## 4. Inspección cualitativa de los registros

Se seleccionaron aleatoriamente algunos registros utilizando una semilla fija para verificar manualmente la relación entre `article` y `abstract`.

Esta revisión busca comprobar que `article` contiene contenido científico y que `abstract` corresponde a una síntesis del contenido asociado.

In [ ]:
pd.set_option("display.max_colwidth", 500)

df_train = dataset["train"].to_pandas()

muestra = df_train.sample(3, random_state=42)

for i, fila in muestra.iterrows():
    print("=" * 100)
    print(f"Índice: {i}")

    print("\nARTICLE:")
    print(fila["article"][:2000])

    print("\nABSTRACT:")
    print(fila["abstract"])

    print("\n")

Índice: 120882

ARTICLE:
two decades ago bender and boettcher have found that a broad family of non - hermitian hamiltonians can exhibit entirely real spectra as long as these hamiltonians have parity - time ( @xmath0 ) symmetry @xcite . 
 one distinguishing feature of @xmath0-symmetric hamiltonians is the existence of spontaneous symmetry breaking , corresponding to a transition from real to complex spectra @xcite . 
 since then , numerious @xmath0-symmetric systems have been explored in several fields , from the complex extension of quantum mechanics @xcite , to the quantum field theories and mathematical physics @xcite , open quantum systems @xcite , the anderson models for disorder systems @xcite , the optical systems with complex refractive indices @xcite , and the topological insulators @xcite .    in recent years , the non - hermitian lattice models with @xmath0 symmetry have been extensively studied , which is stimulated by their experimental realizations in optical waveguides 

In [ ]:
muestra = df_train.sample(5, random_state=42)

for i, fila in muestra.iterrows():
    print("=" * 100)
    print(f"Registro: {i}")
    print(f"Longitud artículo: {len(fila['article'].split())} palabras")
    print(f"Longitud abstract: {len(fila['abstract'].split())} palabras")

    print("\nInicio del artículo:")
    print(fila["article"][:500])

    print("\nAbstract:")
    print(fila["abstract"][:1000])

Registro: 120882
Longitud artículo: 4953 palabras
Longitud abstract: 146 palabras

Inicio del artículo:
two decades ago bender and boettcher have found that a broad family of non - hermitian hamiltonians can exhibit entirely real spectra as long as these hamiltonians have parity - time ( @xmath0 ) symmetry @xcite . 
 one distinguishing feature of @xmath0-symmetric hamiltonians is the existence of spontaneous symmetry breaking , corresponding to a transition from real to complex spectra @xcite . 
 since then , numerious @xmath0-symmetric systems have been explored in several fields , from the compl

Abstract:
we study the effect of @xmath0-symmetric complex potentials on the transport properties of non - hermitian systems , which consist of an infinite linear chain and two side - coupled defect points with @xmath0-symmetric complex on - site potentials . by analytically solving the scattering problem of two typical models , 
 which display standard fano resonances in the absence of non 

### Observaciones

La inspección cualitativa permitió comprobar que los registros contienen textos científicos y que los abstracts presentan una síntesis del contenido desarrollado en los artículos correspondientes.

También se identificó la presencia recurrente de marcadores como `@xmath` y `@xcite`, utilizados para representar expresiones matemáticas y referencias bibliográficas dentro del texto.

## 5. Conclusión del análisis exploratorio

El análisis realizado muestra que el conjunto `ccdv/arxiv-summarization` presenta una estructura consistente, con 215.913 registros distribuidos entre entrenamiento, validación y prueba. No se identificaron valores nulos, aunque se encontraron algunos registros vacíos y duplicados en `train`, los cuales deberán ser considerados durante la preparación de la muestra experimental.

La principal característica relevante para el proyecto corresponde a la extensión de los documentos. Los artículos presentan aproximadamente 5.900–6.000 palabras en promedio, mientras que los abstracts son considerablemente más cortos. Esta diferencia confirma la presencia de entradas extensas que pueden superar las ventanas de contexto de modelos preentrenados convencionales, justificando la evaluación de diferentes estrategias de manejo de contexto largo.

A partir de estos resultados, el dataset se considera adecuado para la experimentación propuesta, siempre que durante la selección de la muestra se excluyan los registros vacíos y los duplicados completos identificados.